# County Health Trends (2019–2023)
## ADS-508 Final Project – Data Cleaning and Exploration

## 1. Setup: Libraries and Tools
We begin by importing libraries for data loading, cloud storage, and data exploration.


In [26]:
!pip install httpimport

In [27]:
import pandas as pd
import boto3 
import httpimport 
import io 

## 2. Load and Combine Multi-Year Datasets
We load public health datasets from 2019–2023 and combine them into one DataFrame for analysis.

In [28]:
# Define file paths for all datasets
files = [
    "/home/sagemaker-user/ADS-508-team-project/datasets/analytic_data2019.csv",
    "/home/sagemaker-user/ADS-508-team-project/datasets/analytic_data2020_0.csv",
    "/home/sagemaker-user/ADS-508-team-project/datasets/analytic_data2021.csv",
    "/home/sagemaker-user/ADS-508-team-project/datasets/analytic_data2022.csv",
    "/home/sagemaker-user/ADS-508-team-project/datasets/analytic_data2023_0.csv"
]

# Define a function to load and clean the dataset
def load_and_clean_csv(file):
    df = pd.read_csv(file, header=0, low_memory=False)
    df = df.drop(index=0).reset_index(drop=True)
    df.columns = df.columns.str.strip().str.lower()
    return df

# Apply the function to all files
df_list = [load_and_clean_csv(file) for file in files]

# Merge all datasets
df_combined = pd.concat(df_list, ignore_index=True)

## 3. Select Relevant Features & Renamed Columns

In [71]:
# Select Relevant Columns
selected_columns = [
   'release year', 'state abbreviation', 'name',
    'premature death raw value', 'poor or fair health raw value', 'adult smoking raw value',
    'adult obesity raw value', 'physical inactivity raw value', 'access to exercise opportunities raw value',
    'flu vaccinations raw value', 'mammography screening raw value', 'uninsured raw value',
    'driving alone to work raw value', 'long commute - driving alone raw value',
    'severe housing problems raw value', 'air pollution - particulate matter raw value',
    'drinking water violations raw value', 'children in poverty raw value', 'income inequality raw value',
    'social associations raw value', 'homeownership raw value', 'median household income raw value',
    'population raw value'
]

#Use selected columns and rename them
df_final = df_combined[selected_columns]

rename_dict = {
    'release year': 'year',
    'state abbreviation': 'state',
    'name': 'county',
    'premature death raw value': 'premature_death',
    'poor or fair health raw value': 'poor_fair_health',
    'adult smoking raw value': 'adult_smoking',
    'adult obesity raw value': 'obesity_rate',
    'physical inactivity raw value': 'physical_inactivity',
    'access to exercise opportunities raw value': 'access_exercise',
    'flu vaccinations raw value': 'flu_vaccination_rate',
    'mammography screening raw value': 'mammography_screening',
    'uninsured raw value': 'uninsured_rate',
    'driving alone to work raw value': 'driving_alone_to_work',
    'long commute - driving alone raw value': 'long_commute',
    'severe housing problems raw value': 'severe_housing_problems',
    'air pollution - particulate matter raw value': 'air_pollution_pmatter',
    'drinking water violations raw value': 'drinking_water_violations',
    'children in poverty raw value': 'children_in_poverty',
    'income inequality raw value': 'income_inequality',
    'social associations raw value': 'social_associations',
    'homeownership raw value': 'homeownership',
    'median household income raw value': 'median_household_income',
    'population raw value': 'population'
}

df_final= df_final.rename(columns=rename_dict)


print(df_final.head())

   year state          county premature_death poor_fair_health adult_smoking  \
0  2019    US   United States     6900.630354              NaN           NaN   
1  2019    AL         Alabama    9917.2328984     0.2140240566   0.215381544   
2  2019    AL  Autauga County    8824.0571232     0.1841112436   0.191246585   
3  2019    AL  Baldwin County    7224.6321603     0.1806045782  0.1679548515   
4  2019    AL  Barbour County     9586.165037     0.2577341563  0.2154087757   

  obesity_rate physical_inactivity access_exercise flu_vaccination_rate  ...  \
0        0.285               0.222    0.8389448174                  NaN  ...   
1        0.351               0.282    0.6164961831                 0.42  ...   
2        0.375               0.311     0.686775027                 0.41  ...   
3         0.31               0.238    0.7197103119                 0.45  ...   
4        0.443               0.282    0.5362566923                 0.37  ...   

  long_commute severe_housing_problems

In [43]:
# Path in SageMaker
cleaned_file_path = "/home/sagemaker-user/ADS-508-team-project/datasets/analytic_data_2019_2023_cleaned.csv"

# Save the cleaned dataset
df_final.to_csv(cleaned_file_path, index=False)

## 4. Exploratory Data Analysis (EDA) 
We use the `jeda` library to generate quick insights and data quality checks. We also used imputation to fill missing values in continous features using the mean.


In [61]:
# Upload dataset to S3
s3 = boto3.client('s3')
bucket_name = 'jc-bucket-ads508-2025-finalproject'
s3_key = "datasets/analytic_data_2019_2023.csv"

s3.upload_file(cleaned_file_path, bucket_name, s3_key)

In [62]:
# import jcds 
with httpimport.github_repo('junclemente', 'jcds', ref='0.1.0'):
    import jcds.eda as jeda


In [63]:
# load dataset from s3 into dataframe
obj = s3.get_object(Bucket=bucket_name, Key=s3_key)
# read Excel file into dataframe
df = pd.read_csv(io.BytesIO(obj['Body'].read()))

print(df.head())

   year state          county  premature_death  poor_fair_health  \
0  2019    US   United States      6900.630354               NaN   
1  2019    AL         Alabama      9917.232898          0.214024   
2  2019    AL  Autauga County      8824.057123          0.184111   
3  2019    AL  Baldwin County      7224.632160          0.180605   
4  2019    AL  Barbour County      9586.165037          0.257734   

   adult_smoking  obesity_rate  physical_inactivity  access_exercise  \
0            NaN         0.285                0.222         0.838945   
1       0.215382         0.351                0.282         0.616496   
2       0.191247         0.375                0.311         0.686775   
3       0.167955         0.310                0.238         0.719710   
4       0.215409         0.443                0.282         0.536257   

   flu_vaccination_rate  ...  long_commute  severe_housing_problems  \
0                   NaN  ...         0.352                 0.183731   
1               

In [64]:
# verify that file exists in s3
response = s3.list_objects_v2(Bucket=bucket_name, Prefix = 'datasets/')
for obj in response.get('Contents', []):
    print(obj['Key'])

datasets/analytic_data_2019_2023.csv
datasets/analytic_data_2019_2023_cleaned.csv
datasets/employee_benefits.xlsx
datasets/employee_benefits_2010_2024.csv
datasets/employee_benefits_file_info.csv


In [65]:
jeda.quick_report(df) 

Quick Report - info(memory_usage='deep')
Total cols: 23
Rows missing all values: 0 (0.0%)
Total Rows: 15970
Cols with missing values: 16 (69.57%)
Total missing values in dataset: 1069


In [66]:
jeda.long_report(df)

Quick Report - info(memory_usage='deep')
Total cols: 23
Rows missing all values: 0 (0.0%)
Total Rows: 15970
Cols with missing values: 16 (69.57%)
Total missing values in dataset: 1069
Categorical features: 2
- state: 52 unique values
- county: 1929 unique values
Continuous features: 21
- year: 5 unique values
- premature_death: 12533 unique values
- poor_fair_health: 9912 unique values
- adult_smoking: 9894 unique values
- obesity_rate: 378 unique values
- physical_inactivity: 379 unique values
- access_exercise: 11596 unique values
- flu_vaccination_rate: 70 unique values
- mammography_screening: 66 unique values
- uninsured_rate: 15929 unique values
- driving_alone_to_work: 15890 unique values
- long_commute: 691 unique values
- severe_housing_problems: 14578 unique values
- air_pollution_pmatter: 171 unique values
- drinking_water_violations: 186 unique values
- children_in_poverty: 509 unique values
- income_inequality: 15948 unique values
- social_associations: 14922 unique values

In [67]:
jeda.dqr_cont(df)

The non-categorical features are: 
['year', 'premature_death', 'poor_fair_health', 'adult_smoking', 'obesity_rate', 'physical_inactivity', 'access_exercise', 'flu_vaccination_rate', 'mammography_screening', 'uninsured_rate', 'driving_alone_to_work', 'long_commute', 'severe_housing_problems', 'air_pollution_pmatter', 'drinking_water_violations', 'children_in_poverty', 'income_inequality', 'social_associations', 'homeownership', 'median_household_income', 'population']
Data Quality for Continous Features
Total Features: 21 / 15970 rows


,Feature,Count,Missing,% missing,Cardinality
0,year,15970,0,0.00,5
1,premature_death,15670,300,1.88,12533
2,poor_fair_health,15967,3,0.02,9912
3,adult_smoking,15967,3,0.02,9894
4,obesity_rate,15968,2,0.01,378
5,physical_inactivity,15968,2,0.01,379
6,access_exercise,15814,156,0.98,11596
7,flu_vaccination_rate,15880,90,0.56,70
8,mammography_screening,15870,100,0.63,66
9,uninsured_rate,15965,5,0.03,15929




Descriptive Stats


,count,mean,std,min,25%,50%,75%,max
year,15970.0,2021.00,1.41,2019.00,2020.00,2021.00,2022.00,2.023000e+03
premature_death,15670.0,8647.89,2818.70,1625.72,6719.42,8298.15,10179.86,4.393907e+04
poor_fair_health,15967.0,0.18,0.05,0.06,0.15,0.18,0.22,4.500000e-01
adult_smoking,15967.0,0.19,0.04,0.06,0.16,0.19,0.22,4.500000e-01
obesity_rate,15968.0,0.34,0.05,0.11,0.31,0.34,0.38,5.900000e-01
physical_inactivity,15968.0,0.27,0.06,0.08,0.23,0.27,0.31,5.200000e-01
access_exercise,15814.0,0.61,0.24,0.00,0.47,0.64,0.79,1.000000e+00
flu_vaccination_rate,15880.0,0.43,0.10,0.02,0.37,0.44,0.50,7.100000e-01
mammography_screening,15870.0,0.40,0.08,0.03,0.35,0.40,0.45,6.900000e-01
uninsured_rate,15965.0,0.12,0.05,0.02,0.08,0.11,0.15,4.100000e-01


In [68]:
# Fill missing values with mean for continous variables
df = df.fillna(df.mean(numeric_only=True))
jeda.dqr_cont(df)

The non-categorical features are: 
['year', 'premature_death', 'poor_fair_health', 'adult_smoking', 'obesity_rate', 'physical_inactivity', 'access_exercise', 'flu_vaccination_rate', 'mammography_screening', 'uninsured_rate', 'driving_alone_to_work', 'long_commute', 'severe_housing_problems', 'air_pollution_pmatter', 'drinking_water_violations', 'children_in_poverty', 'income_inequality', 'social_associations', 'homeownership', 'median_household_income', 'population']
Data Quality for Continous Features
Total Features: 21 / 15970 rows


,Feature,Count,Missing,% missing,Cardinality
0,year,15970,0,0.0,5
1,premature_death,15970,0,0.0,12533
2,poor_fair_health,15970,0,0.0,9912
3,adult_smoking,15970,0,0.0,9894
4,obesity_rate,15970,0,0.0,378
5,physical_inactivity,15970,0,0.0,379
6,access_exercise,15970,0,0.0,11596
7,flu_vaccination_rate,15970,0,0.0,70
8,mammography_screening,15970,0,0.0,66
9,uninsured_rate,15970,0,0.0,15929




Descriptive Stats


,count,mean,std,min,25%,50%,75%,max
year,15970.0,2021.00,1.41,2019.00,2020.00,2021.00,2022.00,2.023000e+03
premature_death,15970.0,8647.89,2792.10,1625.72,6755.35,8365.71,10142.19,4.393907e+04
poor_fair_health,15970.0,0.18,0.05,0.06,0.15,0.18,0.22,4.500000e-01
adult_smoking,15970.0,0.19,0.04,0.06,0.16,0.19,0.22,4.500000e-01
obesity_rate,15970.0,0.34,0.05,0.11,0.31,0.34,0.38,5.900000e-01
physical_inactivity,15970.0,0.27,0.06,0.08,0.23,0.27,0.31,5.200000e-01
access_exercise,15970.0,0.61,0.24,0.00,0.47,0.64,0.79,1.000000e+00
flu_vaccination_rate,15970.0,0.43,0.10,0.02,0.37,0.44,0.50,7.100000e-01
mammography_screening,15970.0,0.40,0.08,0.03,0.35,0.40,0.45,6.900000e-01
uninsured_rate,15970.0,0.12,0.05,0.02,0.08,0.11,0.15,4.100000e-01


In [69]:
jeda.dqr_cat(df)

The categorical features are: 
['state', 'county']
Data Quality Report for Categorical Features
Total features: 2 / 15970 rows
Stats
-----


,Feature,Count,Missing,% Missing,Cardinality
0,state,15970,0,0.0,52
1,county,15970,0,0.0,1929




Mode 1
------


,Feature,Mode 1,Mode 1 Freq.,Mode 1 %
0,state,TX,1275,7.98
1,county,Washington County,150,0.94




Mode 2
------


,Feature,Mode 2,Mode 2 Freq.,Mode 2 %
0,state,GA,800,5.01
1,county,Jefferson County,125,0.78




Descriptive Stats
-----------------


,count,unique,top,freq
state,15970,52,TX,1275
county,15970,1929,Washington County,150


## 5. Save and Upload Cleaned Data to S3
We export the cleaned dataset and upload it to our S3 bucket for storage and future access.


In [70]:
# Save the cleaned dataframe to a local CSV file
df.to_csv("final_cleaned_data.csv", index=False)

# Upload the file to S3
s3.upload_file("final_cleaned_data.csv", bucket_name, "datasets/final_cleaned_data.csv")

print("✅ Cleaned data uploaded to S3!")

✅ Cleaned data uploaded to S3!
